# in-kind LinkML Schema — Entity-Relation Diagram

Interactive graph of classes, attributes, and relations parsed directly from the LinkML YAML schema files.

**Quick start:** run *Setup* cell once → restart kernel → run all remaining cells.

**Controls:**
- Toggle **Show attributes** to display slots as child nodes on each class
- Toggle **Abstract classes** / **Mixin classes** to show/hide those node types
- Click **Expand item subclasses** to unfold the 16 `DonationItem` concrete subclasses
- Use **Filter schemas** (multi-select) to focus on a subset of the schema
- Click any node to see its full slot table in the info panel below the graph
- Change **Layout** to re-arrange (dagre = hierarchical top-down recommended)

## Setup

In [ ]:
# Run once, then restart the kernel
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'ipycytoscape', 'pyyaml', 'ipywidgets', '--quiet'],
               check=True)

## Parse schema files

In [1]:
import yaml, pathlib, json, re
from collections import defaultdict
import ipycytoscape
import ipywidgets as widgets
from IPython.display import display, HTML

In [8]:

# ── Schema parser ──────────────────────────────────────────────────────────
# Point SCHEMA_DIR at your inkind-knowledge-repo src directory.
# The list below uses the files from the project folder directly.

import os

SCHEMA_FILES = [
    "actor.yaml", "organisation.yaml", "donation_source.yaml",
    "donation_collection.yaml", "donation_item.yaml", "storage_location.yaml",
    "demand_signal.yaml", "campaign.yaml", "provenance.yaml",
    "step_type.yaml", "process_template.yaml", "fragment_binding.yaml",
]

full_path = os.path.join("C:/","in-kind","inkind-knowledge-repo","src", "inkind_knowledge_repo")

# Try project path first, then working directory
def find_file(name):
    candidates = [
        os.path.join(full_path, name),
        f"./{name}",
        os.path.join(full_path, "schema", name),
        os.path.join(full_path, "schema", "entities", name),
        os.path.join(full_path, "schema", "process", name),
        os.path.join(full_path, "schema", "ui", name),
        os.path.join(full_path, "schema", "categories", name),
        os.path.join(full_path, "schema", "states", name),
        f"../schema/entities/{name}",
        f"../schema/process/{name}",
        f"../schema/ui/{name}",
        f"../schema/{name}",
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return None

classes  = {}   # name -> {source, is_a, mixins, abstract, mixin, slots, slot_usage, desc}
slots_db = {}   # name -> {range, required, multivalued, desc}

for fname in SCHEMA_FILES:
    fpath = find_file(fname)
    if not fpath:
        print(f"  ⚠ not found: {fname}")
        continue
    print(f"  ✓ Found: {fpath}")
    with open(fpath, encoding="utf-8") as f:
        schema = yaml.safe_load(f)
    src = pathlib.Path(fname).stem

    for cname, cdef in (schema.get("classes") or {}).items():
        cdef = cdef or {}
        classes[cname] = {
            "source": src,
            "is_a":   cdef.get("is_a"),
            "mixins": cdef.get("mixins") or [],
            "abstract": bool(cdef.get("abstract")),
            "mixin":    bool(cdef.get("mixin")),
            "slots":    cdef.get("slots") or [],
            "slot_usage": cdef.get("slot_usage") or {},
            "desc":     (cdef.get("description") or "").replace("\n"," ")[:90],
        }

    for sname, sdef in (schema.get("slots") or {}).items():
        sdef = sdef or {}
        slots_db[sname] = {
            "range":       sdef.get("range", "string"),
            "required":    bool(sdef.get("required")),
            "multivalued": bool(sdef.get("multivalued")),
            "desc":        (sdef.get("description") or "").split("\n")[0][:70],
        }

entity_names = set(classes.keys())

# Resolve effective range for a slot on a class (slot_usage overrides global)
def effective_range(cname, sname):
    su = (classes[cname]["slot_usage"].get(sname) or {})
    return su.get("range") or (slots_db.get(sname) or {}).get("range", "string")

def effective_required(cname, sname):
    su = (classes[cname]["slot_usage"].get(sname) or {})
    return su.get("required", (slots_db.get(sname) or {}).get("required", False))

def effective_multivalued(sname):
    return (slots_db.get(sname) or {}).get("multivalued", False)

# ── Build relation index ──────────────────────────────────────────────────
relations = []   # (source_class, slot_name, target_class, required, multivalued)
for cname, cdef in classes.items():
    for sname in cdef["slots"]:
        rng = effective_range(cname, sname)
        if rng in entity_names:
            relations.append({
                "source": cname, "slot": sname, "target": rng,
                "required": effective_required(cname, sname),
                "multivalued": effective_multivalued(sname),
            })

# ── Build inheritance index ──────────────────────────────────────────────
inheritance = []  # (child, parent, edge_type)  edge_type: is_a | mixin
for cname, cdef in classes.items():
    if cdef["is_a"] and cdef["is_a"] in entity_names:
        inheritance.append({"child": cname, "parent": cdef["is_a"], "type": "is_a"})
        
    for mx in cdef["mixins"]:
        if mx in entity_names:
            inheritance.append({"child": cname, "parent": mx, "type": "mixin"})            

print(f"✓ Parsed {len(classes)} classes, {len(slots_db)} slots")
print(f"  Relations:   {len(relations)}")
print(f"  Inheritance: {len(inheritance)}")


  ✓ Found: C:/in-kind\inkind-knowledge-repo\src\inkind_knowledge_repo\schema\entities\actor.yaml
  ✓ Found: C:/in-kind\inkind-knowledge-repo\src\inkind_knowledge_repo\schema\entities\organisation.yaml
  ✓ Found: C:/in-kind\inkind-knowledge-repo\src\inkind_knowledge_repo\schema\entities\donation_source.yaml
  ✓ Found: C:/in-kind\inkind-knowledge-repo\src\inkind_knowledge_repo\schema\entities\donation_collection.yaml
  ✓ Found: C:/in-kind\inkind-knowledge-repo\src\inkind_knowledge_repo\schema\entities\donation_item.yaml
  ✓ Found: C:/in-kind\inkind-knowledge-repo\src\inkind_knowledge_repo\schema\entities\storage_location.yaml
  ✓ Found: C:/in-kind\inkind-knowledge-repo\src\inkind_knowledge_repo\schema\entities\demand_signal.yaml
  ✓ Found: C:/in-kind\inkind-knowledge-repo\src\inkind_knowledge_repo\schema\entities\campaign.yaml
  ✓ Found: C:/in-kind\inkind-knowledge-repo\src\inkind_knowledge_repo\schema\provenance.yaml
  ✓ Found: ./step_type.yaml
  ✓ Found: ./process_template.yaml
  ✓ Fou

## Visual configuration

In [9]:
# ── Visual grouping / colouring by source schema ──────────────────────────
GROUP_COLORS = {
    "donation_item":       {"bg": "#E1F5EE", "border": "#1D9E75", "label_color": "#085041"},
    "actor":               {"bg": "#EEEDFE", "border": "#7F77DD", "label_color": "#26215C"},
    "organisation":        {"bg": "#EEEDFE", "border": "#7F77DD", "label_color": "#26215C"},
    "donation_collection": {"bg": "#E6F1FB", "border": "#378ADD", "label_color": "#042C53"},
    "donation_source":     {"bg": "#E6F1FB", "border": "#378ADD", "label_color": "#042C53"},
    "demand_signal":       {"bg": "#FAEEDA", "border": "#BA7517", "label_color": "#412402"},
    "campaign":            {"bg": "#FAEEDA", "border": "#BA7517", "label_color": "#412402"},
    "storage_location":    {"bg": "#EAF3DE", "border": "#639922", "label_color": "#173404"},
    "provenance":          {"bg": "#FAECE7", "border": "#D85A30", "label_color": "#4A1B0C"},
    "step_type":           {"bg": "#FBEAF0", "border": "#D4537E", "label_color": "#4B1528"},
    "process_template":    {"bg": "#FBEAF0", "border": "#D4537E", "label_color": "#4B1528"},
    "fragment_binding":    {"bg": "#F1EFE8", "border": "#888780", "label_color": "#2C2C2A"},
}
DEFAULT_COLORS = {"bg": "#F1EFE8", "border": "#888780", "label_color": "#2C2C2A"}

def get_colors(cname):
    src = classes[cname]["source"]
    return GROUP_COLORS.get(src, DEFAULT_COLORS)

# Slot type categories for attribute display
PRIMITIVE_RANGES = {
    "string","integer","float","boolean","date","datetime",
    "uri","uriorcurie","Any","curie","ncname","jsonpointer","sparqlpath"
}

def is_primitive(range_str):
    if not range_str:
        return True
    return (range_str in PRIMITIVE_RANGES
            or range_str.endswith("Enum")
            or range_str.lower() in {"string","int","float","bool","date","datetime"})

def slot_display_type(sname, rng):
    """Short display label for a slot's type."""
    if rng and rng.endswith("Enum"):
        return rng.replace("Enum","") + " (enum)"
    if rng in PRIMITIVE_RANGES:
        return rng
    return rng or "string"

print("Groups defined:", list(GROUP_COLORS.keys()))


Groups defined: ['donation_item', 'actor', 'organisation', 'donation_collection', 'donation_source', 'demand_signal', 'campaign', 'storage_location', 'provenance', 'step_type', 'process_template', 'fragment_binding']


In [10]:
# ── Cytoscape style definitions ──────────────────────────────────────────

CY_STYLE = [
    # ── Base node ──────────────────────────────────────────────────────────
    {"selector": "node", "css": {
        "label":           "data(label)",
        "text-valign":     "center",
        "text-halign":     "center",
        "text-wrap":       "wrap",
        "text-max-width":  "140px",
        "font-size":       "11px",
        "font-family":     "monospace",
        "width":           "label",
        "height":          "label",
        "padding":         "8px",
        "shape":           "roundrectangle",
        "background-color":"data(bg)",
        "border-width":    "2px",
        "border-color":    "data(border)",
        "color":           "data(lcolor)",
    }},

    # ── Abstract class ─────────────────────────────────────────────────────
    {"selector": "node[type='class_abs']", "css": {
        "border-style":    "dashed",
        "font-style":      "italic",
    }},

    # ── Mixin class ────────────────────────────────────────────────────────
    {"selector": "node[type='class_mixin']", "css": {
        "border-style":    "dotted",
        "opacity":         "0.85",
    }},

    # ── Attribute node ─────────────────────────────────────────────────────
    {"selector": "node[type='attr']", "css": {
        "shape":           "rectangle",
        "font-size":       "9px",
        "font-family":     "monospace",
        "border-width":    "1px",
        "border-style":    "solid",
        "opacity":         "0.9",
        "padding":         "4px",
        "text-valign":     "center",
        "text-halign":     "left",
        "text-margin-x":   "4px",
    }},
    {"selector": "node[type='attr'][?required]", "css": {
        "font-weight":     "bold",
    }},

    # ── Edges ──────────────────────────────────────────────────────────────
    {"selector": "edge", "css": {
        "curve-style":          "bezier",
        "width":                "1.5px",
        "line-color":           "#aaa",
        "target-arrow-shape":   "triangle",
        "target-arrow-color":   "#aaa",
        "font-size":            "9px",
        "color":                "#666",
        "label":                "data(label)",
        "text-rotation":        "autorotate",
        "text-margin-y":        "-6px",
        "text-background-color":"#fff",
        "text-background-opacity":"0.7",
        "text-background-padding":"1px",
    }},

    # relation edges
    {"selector": "edge[type='relation']", "css": {
        "line-color":          "#378ADD",
        "target-arrow-color":  "#378ADD",
        "color":               "#185FA5",
    }},
    {"selector": "edge[type='relation'][?required]", "css": {
        "width": "2.5px",
    }},
    {"selector": "edge[type='relation'][?multivalued]", "css": {
        "target-arrow-shape": "triangle",
        "mid-target-arrow-shape": "none",
        "line-style": "solid",
    }},

    # is_a edges
    {"selector": "edge[type='is_a']", "css": {
        "line-color":          "#1D9E75",
        "target-arrow-color":  "#1D9E75",
        "target-arrow-shape":  "triangle-hollow",
        "line-style":          "solid",
        "width":               "1.5px",
        "color":               "#085041",
    }},

    # mixin edges
    {"selector": "edge[type='mixin']", "css": {
        "line-color":          "#7F77DD",
        "target-arrow-color":  "#7F77DD",
        "target-arrow-shape":  "diamond",
        "line-style":          "dotted",
        "width":               "1.5px",
        "color":               "#534AB7",
    }},

    # attribute connector edges (thin, no label, grey)
    {"selector": "edge[type='has_attr']", "css": {
        "line-color":          "#ddd",
        "target-arrow-shape":  "none",
        "line-style":          "solid",
        "width":               "0.8px",
        "label":               "",
    }},

    # ── Selection ──────────────────────────────────────────────────────────
    {"selector": ":selected", "css": {
        "border-color": "#D85A30",
        "border-width":  "3px",
        "overlay-color": "#D85A30",
        "overlay-opacity": "0.1",
    }},
]

print("Styles defined")


Styles defined


## Graph builder

In [11]:
# ── Graph element builder ─────────────────────────────────────────────────

def build_elements(
    show_attrs=True,
    show_abstract=True,
    show_mixins=True,
    show_subclasses=True,   # whether DonationItem subclasses are expanded
    filter_sources=None,    # None = all; set of source names to include
):
    """
    Returns (nodes, edges) for ipycytoscape.

    Node types:
      class       – regular class node
      class_abs   – abstract class
      class_mixin – mixin class
      attr        – attribute node (child of class, shown when show_attrs=True)

    Edge types:
      relation    – slot pointing to another class
      is_a        – inheritance
      mixin_use   – mixin application
      has_attr    – class → attribute node
    """
    nodes, edges = [], []
    included = set()

    # ── Determine which classes to include ───────────────────────────────
    for cname, cdef in classes.items():
        if filter_sources and cdef["source"] not in filter_sources:
            continue
        if not show_abstract and cdef["abstract"]:
            continue
        if not show_mixins and cdef["mixin"]:
            continue
        # If not expanding subclasses, skip concrete DonationItem subclasses
        if not show_subclasses and cdef["is_a"] == "DonationItem":
            continue
        included.add(cname)

    # Ensure parents of included classes are also included (for inheritance edges)
    to_add = set()
    for cname in included:
        parent = classes[cname].get("is_a")
        if parent and parent in classes and parent not in included:
            to_add.add(parent)
    included |= to_add

    # ── Class nodes ──────────────────────────────────────────────────────
    for cname in included:
        cdef = classes[cname]
        col  = get_colors(cname)
        ctype = ("class_abs" if cdef["abstract"]
                 else "class_mixin" if cdef["mixin"]
                 else "class")

        # Build attribute label list (primitive-range slots only)
        attr_lines = []
        for sname in cdef["slots"]:
            rng = effective_range(cname, sname)
            if not is_primitive(rng):
                continue
            req  = effective_required(cname, sname)
            multi= effective_multivalued(sname)
            dtype= slot_display_type(sname, rng)
            marker = "* " if req else "  "
            mv_mark = "[]" if multi else ""
            attr_lines.append(f"{marker}{sname}: {dtype}{mv_mark}")

        attrs_text = "\n".join(attr_lines) if attr_lines else ""

        nodes.append({"data": {
            "id":         cname,
            "label":      cname,
            "type":       ctype,
            "source":     cdef["source"],
            "desc":       cdef["desc"],
            "attr_count": len(attr_lines),
            "attrs_text": attrs_text,
            "bg":         col["bg"],
            "border":     col["border"],
            "lcolor":     col["label_color"],
            "expanded":   True,   # toggled by user
        }})

    # ── Attribute nodes (separate nodes per class if show_attrs) ─────────
    if show_attrs:
        for cname in included:
            cdef = classes[cname]
            col  = get_colors(cname)
            for sname in cdef["slots"]:
                rng = effective_range(cname, sname)
                if not is_primitive(rng):
                    continue
                req  = effective_required(cname, sname)
                multi= effective_multivalued(sname)
                dtype= slot_display_type(sname, rng)
                aid = f"{cname}__attr__{sname}"
                nodes.append({"data": {
                    "id":     aid,
                    "label":  f"{sname}: {dtype}" + ("[]" if multi else ""),
                    "type":   "attr",
                    "parent_class": cname,
                    "required": req,
                    "bg":    col["bg"],
                    "border": col["border"],
                }})
                edges.append({"data": {
                    "id": f"e_attr_{cname}_{sname}",
                    "source": cname, "target": aid,
                    "type": "has_attr",
                }})

    # ── Relation edges ───────────────────────────────────────────────────
    for rel in relations:
        if rel["source"] not in included or rel["target"] not in included:
            continue
        edges.append({"data": {
            "id":         f"rel_{rel['source']}_{rel['slot']}",
            "source":     rel["source"],
            "target":     rel["target"],
            "type":       "relation",
            "label":      rel["slot"] + ("[]" if rel["multivalued"] else ""),
            "required":   rel["required"],
            "multivalued":rel["multivalued"],
        }})

    # ── Inheritance / mixin edges ────────────────────────────────────────
    for inh in inheritance:
        if inh["child"] not in included or inh["parent"] not in included:
            continue
        edges.append({"data": {
            "id":     f"inh_{inh['child']}_{inh['parent']}",
            "source": inh["child"],
            "target": inh["parent"],
            "type":   inh["type"],   # is_a or mixin
            "label":  "",
        }})

    return nodes, edges

print("Builder ready — call build_elements() to construct the graph")


Builder ready — call build_elements() to construct the graph


## Interactive ERD viewer

In [15]:
# ── Main interactive viewer ──────────────────────────────────────────────

# ── Controls ──────────────────────────────────────────────────────────────
all_sources = sorted(set(c["source"] for c in classes.values()))

w_attrs      = widgets.ToggleButton(value=False, description="Show attributes",
                                    button_style="", layout=widgets.Layout(width="160px"))
w_abstract   = widgets.ToggleButton(value=True,  description="Abstract classes",
                                    button_style="", layout=widgets.Layout(width="150px"))
w_mixins     = widgets.ToggleButton(value=False, description="Mixin classes",
                                    button_style="", layout=widgets.Layout(width="140px"))
w_subclasses = widgets.ToggleButton(value=False, description="Item subclasses",
                                    button_style="info", layout=widgets.Layout(width="160px"))
w_layout     = widgets.Dropdown(
    options=["dagre","breadthfirst","cose","grid","circle"],
    value="dagre", description="Layout:", layout=widgets.Layout(width="180px"))
w_sources    = widgets.SelectMultiple(
    options=all_sources, value=all_sources,
    description="Schemas:", rows=min(len(all_sources), 8),
    layout=widgets.Layout(width="220px", height="180px"))

info_out = widgets.Output()
cy_out   = widgets.Output()

def build_cy():
    fs = set(w_sources.value) if w_sources.value else None
    nodes, edges = build_elements(
        show_attrs=w_attrs.value,
        show_abstract=w_abstract.value,
        show_mixins=w_mixins.value,
        show_subclasses=w_subclasses.value,
        filter_sources=fs,
    )
    cy = ipycytoscape.CytoscapeWidget()
    for _n in nodes:
        cy.graph.add_node(ipycytoscape.Node(data=_n["data"]))
    for _e in edges:
        cy.graph.add_edge(ipycytoscape.Edge(data=_e["data"]))
    cy.set_style(CY_STYLE)

    layout_opts = {"name": w_layout.value, "padding": 30}
    if w_layout.value == "dagre":
        layout_opts.update({"rankDir": "TB", "rankSep": 80, "nodeSep": 40, "edgeSep": 10})
    elif w_layout.value == "breadthfirst":
        layout_opts.update({"directed": True, "spacingFactor": 1.4})
    elif w_layout.value == "cose":
        layout_opts.update({"nodeRepulsion": 4000, "idealEdgeLength": 80, "gravity": 0.3})
    cy.set_layout(**layout_opts)

    # ── Node click → info panel ────────────────────────────────────────
    def on_node_click(node):
        d = node["data"]
        with info_out:
            info_out.clear_output()
            if d.get("type") == "attr":
                pc = d.get("parent_class","")
                req_str = " <b style='color:#D85A30'>required</b>" if d.get("required") else ""
                display(HTML(
                    f"<b>Attribute:</b> {d['label']}{req_str}<br>"
                    f"<b>On class:</b> {pc}"
                ))
            else:
                cname = d["id"]
                cdef  = classes.get(cname, {})
                col   = get_colors(cname)
                isa   = f" → <i>{cdef.get('is_a')}</i>" if cdef.get("is_a") else ""
                mx    = ", ".join(cdef.get("mixins") or [])
                src   = cdef.get("source","")

                # Slot table
                slot_rows = []
                for sname in (cdef.get("slots") or []):
                    rng  = effective_range(cname, sname)
                    req  = effective_required(cname, sname)
                    multi= effective_multivalued(sname)
                    desc = (slots_db.get(sname) or {}).get("desc","")
                    rel_link = f" → <b>{rng}</b>" if rng in entity_names else ""
                    req_mark = "●" if req else "○"
                    multi_mark = "[]" if multi else ""
                    slot_rows.append(
                        f"<tr>"
                        f"<td style='padding:2px 6px;color:{'#D85A30' if req else '#888'}'>{req_mark}</td>"
                        f"<td style='padding:2px 6px;font-family:monospace'>{sname}{multi_mark}</td>"
                        f"<td style='padding:2px 6px;color:#185FA5'>{rng}{rel_link}</td>"
                        f"<td style='padding:2px 6px;font-size:10px;color:#666'>{desc[:50]}</td>"
                        f"</tr>"
                    )
                slot_table = (
                    "<table style='border-collapse:collapse;font-size:11px;margin-top:6px;width:100%'>"
                    "<tr style='background:#f5f5f5'><th style='padding:2px 6px'>req</th>"
                    "<th style='padding:2px 6px;text-align:left'>slot</th>"
                    "<th style='padding:2px 6px;text-align:left'>range</th>"
                    "<th style='padding:2px 6px;text-align:left'>description</th></tr>"
                    + "".join(slot_rows) + "</table>"
                ) if slot_rows else "<i style='color:#aaa'>no slots</i>"

                abstract_badge = (" <span style='background:#eee;padding:1px 6px;border-radius:8px;font-size:10px'>abstract</span>"
                                  if cdef.get("abstract") else "")
                mixin_badge    = (" <span style='background:#EEEDFE;padding:1px 6px;border-radius:8px;font-size:10px'>mixin</span>"
                                  if cdef.get("mixin") else "")
                display(HTML(
                    f"<div style='font-size:13px'>"
                    f"<b style='color:{col['label_color']}'>{cname}</b>{abstract_badge}{mixin_badge}{isa}<br>"
                    f"<span style='font-size:11px;color:#888'>schema: {src}"
                    + (f" | mixins: {mx}" if mx else "") + "</span><br>"
                    f"<span style='font-size:11px'>{cdef.get('desc','')}</span>"
                    f"</div>"
                    + slot_table
                ))

    # cy.on("tapNode", on_node_click)
    cy.on("node", "click", on_node_click)
    return cy, nodes, edges

# ── Collapse/expand DonationItem subclasses button ────────────────────────
collapse_btn = widgets.Button(description="Expand item subclasses ▸",
                               layout=widgets.Layout(width="200px"))

def on_collapse(b):
    w_subclasses.value = not w_subclasses.value
    collapse_btn.description = ("Collapse item subclasses ▾"
                                 if w_subclasses.value else "Expand item subclasses ▸")
    refresh(None)

collapse_btn.on_click(on_collapse)

# ── Main refresh ──────────────────────────────────────────────────────────
def refresh(_):
    with cy_out:
        cy_out.clear_output()
        cy, n, e = build_cy()
        class_nodes = [x for x in n if x["data"].get("type","").startswith("class")]
        attr_nodes  = [x for x in n if x["data"].get("type") == "attr"]
        rel_edges   = [x for x in e if x["data"].get("type") == "relation"]
        display(HTML(
            f"<span style='font-size:11px;color:#888'>"
            f"{len(class_nodes)} classes &nbsp;|&nbsp; "
            f"{len(attr_nodes)} attribute nodes &nbsp;|&nbsp; "
            f"{len(rel_edges)} relations</span>"
        ))
        display(cy)

for w in [w_attrs, w_abstract, w_mixins, w_layout]:
    w.observe(refresh, names="value")
w_sources.observe(refresh, names="value")

# ── Layout ────────────────────────────────────────────────────────────────
controls_top = widgets.HBox([
    w_attrs, w_abstract, w_mixins, collapse_btn, w_layout
], layout=widgets.Layout(gap="8px", flex_wrap="wrap"))

main = widgets.HBox([
    widgets.VBox([
        widgets.Label("Filter schemas:"),
        w_sources,
        widgets.HTML("""
<div style="font-size:10px;color:#888;margin-top:8px;line-height:1.6">
<b>Legend</b><br>
─── relation<br>
──▷ is_a (hollow)<br>
···◇ mixin<br>
── attr connector<br><br>
<b>●</b> required slot<br>
<b>○</b> optional slot<br>
<i>italic</i> = abstract<br>
dotted = mixin class
</div>"""),
    ], layout=widgets.Layout(width="200px", flex_shrink="0")),
    widgets.VBox([
        controls_top,
        cy_out,
        info_out,
    ], layout=widgets.Layout(flex="1")),
], layout=widgets.Layout(gap="16px"))

display(main)
refresh(None)


## Export

In [ ]:
# ── Optional: Export graph as JSON for external tools ────────────────────
# Run this cell to get Cytoscape.js-compatible JSON

nodes, edges = build_elements(
    show_attrs=True, show_abstract=True, show_mixins=True, show_subclasses=True
)
export = {"nodes": [n["data"] for n in nodes], "edges": [e["data"] for e in edges]}
print(f"Nodes: {len(nodes)}  Edges: {len(edges)}")

# Uncomment to write to file:
# import json
# with open("inkind_erd_graph.json", "w") as f:
#     json.dump(export, f, indent=2)
# print("Written: inkind_erd_graph.json")
